> **Versão pública (2026).** Notebook original, executado no Google Colab em 2022, com os dados numa pasta do Google Drive (`<PASTA_DO_PROJETO>`). Só ficaram as células da estimativa por estado, e as saídas foram limpas, porque eram testes com percentuais digitados à mão. Veja o [README](../README.md).

In [ ]:
!pip install --upgrade pandas

In [ ]:
import os
import shutil
from google.colab import drive

import pandas as pd
import numpy as np

import random
from IPython.display import clear_output

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive.mount('/content/drive')

# Lê Dados das Votações por Seção Eleitoral

In [ ]:
#Arquivos baixados do site do TSE
#https://dadosabertos.tse.jus.br/dataset/resultados-2022
#https://dadosabertos.tse.jus.br/dataset/resultados-2022-boletim-de-urna

#Outra Opção de Fonte:
#https://basedosdados.org/dataset/br-tse-eleicoes?bdm_table=resultados_candidato_secao

path_dados = '<PASTA_DO_PROJETO>/dados_votacao'
lista_arquivos = ['detalhe_votacao_secao_2022.zip', 'votacao_secao_2022_BR.zip']
print(lista_arquivos)

In [ ]:
#Descompacta os arquivos
for arq in lista_arquivos:
    pasta = arq.split('.')[0]
    os.system(f'unzip {path_dados}/{arq} -d {pasta}')

In [ ]:
def leitura_dados(pasta, arq_csv):
    return pd.read_csv(f'{pasta}/{arq_csv}', sep = ';', engine = 'c', encoding = 'latin-1')

arq = 'detalhe_votacao_secao_2022.zip'
pasta = arq.split('.')[0]
arq_csv = [f for f in os.listdir(pasta) if f.find('.csv') != -1 and f.find('_BRASIL') != -1][0]
df_desc_inp = leitura_dados(pasta, arq_csv)

arq = 'votacao_secao_2022_BR.zip'
pasta = arq.split('.')[0]
arq_csv = [f for f in os.listdir(pasta) if f.find('.csv') != -1][0]
df_inp = leitura_dados(pasta, arq_csv)

In [ ]:
def trata_detalhe(df_desc_inp):
    df_desc = df_desc_inp[df_desc_inp['DS_CARGO'] == 'PRESIDENTE']
    cols_id = ['SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']
    cols_info = ['QT_APTOS', 'QT_COMPARECIMENTO', 'QT_ABSTENCOES', 'QT_VOTOS_NOMINAIS', 'QT_VOTOS_BRANCOS', 'QT_VOTOS_NULOS']
    df_desc = df_desc[np.append(cols_id, cols_info)].reset_index(drop = True)
    return df_desc.reset_index().rename({'index': 'ID_URNA'}, axis = 1).copy()

def trata_votacao(df_inp, df_desc):
    df = df_inp[df_inp['DS_CARGO'] == 'PRESIDENTE']
    cols_id = ['SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'NM_VOTAVEL']
    cols_info = ['QT_VOTOS']
    df = df[np.append(cols_id, cols_info)].reset_index(drop = True).copy()
    cols_urna = ['ID_URNA', 'SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO']
    return pd.merge(df, df_desc[cols_urna], how = 'left', on = ['SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO'])

In [ ]:
df_desc = trata_detalhe(df_desc_inp)
df_desc.sample(n = 5).T

In [ ]:
df = trata_votacao(df_inp, df_desc)
df.sample(n = 5).T

# Resultados da Eleição

In [ ]:
df_res = df[['NM_VOTAVEL', 'QT_VOTOS']].groupby('NM_VOTAVEL').sum().sort_values('QT_VOTOS', ascending = False)
df_res['PERC_VOTOS'] = df_res['QT_VOTOS']*100/df_res['QT_VOTOS'].sum()
df_res['PERC_VOTOS_VALIDOS'] = df_res['QT_VOTOS']*100/df_res.loc[~df_res.index.isin(['VOTO NULO', 'VOTO BRANCO']), 'QT_VOTOS'].sum()
display(df_res)

# Simulador do Segundo Turno

In [ ]:
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets

In [ ]:
df_uf_aptos = df_desc[['SG_UF', 'QT_APTOS']].groupby('SG_UF').sum()

In [ ]:
df_apuracao_realtime = pd.DataFrame(index = df_uf_aptos.index, columns = ['Lula', 'Bolsonaro', 'Brancos', 'Nulos'])
df_apuracao_realtime = df_apuracao_realtime.fillna(0)

In [ ]:
lista_dash = []

grid = widgets.GridspecLayout(df_apuracao_realtime.shape[0], df_apuracao_realtime.shape[1] + 1)
for i in range(df_apuracao_realtime.shape[0]):
    grid[i, 0] = widgets.Label(df_apuracao_realtime.index[i])
    for j in range(df_apuracao_realtime.shape[1]):
        grid[i, j+1] = widgets.FloatText(value = 0, description = df_apuracao_realtime.columns[j])
lista_dash.append(grid)

btn = widgets.Button(description = 'Atualizar')
def btn_eventhandler(obj):
    for i in range(df_apuracao_realtime.shape[0]):
        for j in range(df_apuracao_realtime.shape[1]):
            df_apuracao_realtime.iloc[i, j] = grid[i, j+1].value
btn.on_click(btn_eventhandler)
lista_dash.append(btn)

dashboard = widgets.VBox(lista_dash)
display(dashboard)

In [ ]:
def calcula_estimativa(df_uf_aptos, df_apuracao_realtime):
    df_estimativa = pd.concat([df_uf_aptos, df_apuracao_realtime], axis = 1)
    for c in df_apuracao_realtime.columns:
        df_estimativa[c] = df_estimativa[c]*df_estimativa['QT_APTOS']/100
    df_estimativa = df_estimativa.drop('QT_APTOS', axis = 1)
    df_soma = pd.DataFrame(df_estimativa.sum(axis = 0), columns = ['QT_VOTOS_ESTIMADO'])
    qtd_tot = df_soma['QT_VOTOS_ESTIMADO'].sum()
    df_soma = df_soma*100/qtd_tot
    display(df_soma)

calcula_estimativa(df_uf_aptos, df_apuracao_realtime)